# 03.3 Comments and Docstrings

Comments and docstrings look similar — both are human-readable text inside code —
but they are fundamentally different things. A comment is **thrown away by the
compiler**. A docstring is **stored in the object** and can be read at runtime.

That distinction drives everything about when to use which.

## Theory

### Comments: stripped at compile time

Anything after a `#` is ignored by the tokenizer. It never reaches the bytecode,
costs nothing at runtime, and cannot be read by your program.

```python
x = 5  # this text does not exist in the compiled output
```

There is only one comment syntax in Python. There is no `/* ... */` block
comment — multiple comment lines each need their own `#`.

### Docstrings: stored as data

A **docstring** is a string literal placed as the *first statement* of a module,
function, class, or method. Python stores it in that object's `__doc__`
attribute.

```python
def greet():
    """Say hello."""     # <- stored, readable at runtime
    # this is a comment     # <- discarded
```

This is why `help()` works, why editors show popups, and why documentation tools
can generate reference pages automatically.

### The rule for choosing

- **Comment** = explains *why* this code exists, to whoever edits it next
- **Docstring** = explains *what* this thing does and how to use it, to whoever
  calls it

A caller should never need to read your function body. That is what the docstring
is for.

### The one thing that makes a comment bad

A comment that restates the code adds nothing and **rots** — the code changes,
the comment does not, and now it actively lies.

```python
# Set the rate to 0.18          <- worthless, and will become wrong
rate = 0.18

# Finance confirmed this on 2026-01-15; review each April.
rate = 0.18                     <- says something the code cannot
```

In [ ]:
# Proof that comments vanish and docstrings survive.
NEWLINE = chr(10)

# Build the two sources. QUOTES holds a triple-quote so we can put a
# docstring inside a string without the delimiters clashing.
QUOTES = chr(34) * 3

source_with_comment = (
    "def add(a, b):" + NEWLINE
    + "    # This comment will not appear in the bytecode." + NEWLINE
    + "    return a + b" + NEWLINE
)

source_with_docstring = (
    "def add(a, b):" + NEWLINE
    + "    " + QUOTES + "Add two numbers together." + QUOTES + NEWLINE
    + "    return a + b" + NEWLINE
)

comment_code = compile(source_with_comment, "<demo>", "exec")
docstring_code = compile(source_with_docstring, "<demo>", "exec")

# co_consts holds the constants Python stored. The function object is
# nested inside, so look at its constants too.
print("Constants kept from the COMMENT version:")
for constant in comment_code.co_consts[0].co_consts:
    print("   ", repr(constant))

print("")
print("Constants kept from the DOCSTRING version:")
for constant in docstring_code.co_consts[0].co_consts:
    print("   ", repr(constant))

print("")
print("The comment is gone entirely. The docstring is stored as data.")

## Reading docstrings at runtime

Because docstrings are stored on the object, your program can read them. This is
exactly what `help()` does.

In [ ]:
def calculate_discount(price, percentage):
    """Apply a percentage discount to a price.

    Args:
        price: The original price.
        percentage: Discount as a whole number, so 20 means 20 percent.

    Returns:
        The price after the discount is applied.
    """
    # Convert the percentage into a multiplier and apply it.
    return price * (1 - percentage / 100)


# The docstring is a normal attribute you can read and manipulate.
print("Full docstring:")
print(calculate_discount.__doc__)

print("")
# Take just the summary line - the first line of the docstring.
summary = calculate_discount.__doc__.strip().splitlines()[0]
print("Summary line only:", summary)

print("")
print("Length in characters:", len(calculate_discount.__doc__))

# Built-in objects carry docstrings too.
print("")
print("Built-ins have them as well:")
print("   len:   ", len.__doc__.splitlines()[0])
print("   sorted:", sorted.__doc__.splitlines()[0])

## The four places a docstring can go

A docstring must be the **first statement**. Put it anywhere else and it becomes
an ordinary string expression that is computed and discarded.

In [ ]:
class Account:
    """A bank account with a balance.

    This is a CLASS docstring - it describes what the class represents.
    """

    def __init__(self, owner, balance=0):
        """Set up a new account.

        This is a METHOD docstring.
        """
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        """Add money to the account and return the new balance."""
        self.balance += amount
        return self.balance


def standalone_function():
    """This is a FUNCTION docstring."""
    return True


# Read each one back.
print("Class docstring:")
print("   ", Account.__doc__.strip().splitlines()[0])

print("")
print("Method docstring:")
print("   ", Account.deposit.__doc__)

print("")
print("Function docstring:")
print("   ", standalone_function.__doc__)

print("")
print("A MODULE docstring is the same idea at the top of a .py file.")
print("Notebooks have no module docstring, so here __doc__ is:", repr(__doc__))

### Position matters

Move the string off the first line and Python no longer treats it as a docstring.

In [ ]:
def correct_placement():
    """This IS a docstring - first statement in the function."""
    value = 1
    return value


def wrong_placement():
    value = 1
    """This is NOT a docstring - it is just a discarded string."""
    return value


print("correct_placement.__doc__:", repr(correct_placement.__doc__))
print("wrong_placement.__doc__:  ", repr(wrong_placement.__doc__))

print("")
print("The second string was evaluated and thrown away, exactly like")
print("the bare expression statements you met in 03.1.")

## Triple-quoted strings are not block comments

Python has no block comment syntax. People often use a triple-quoted string
instead:

```python
"""
This looks like a block comment.
"""
```

It works in the sense that nothing breaks — but it is an **expression statement**.
The string is built at runtime and immediately discarded. A real comment costs
nothing at all.

For a few lines it makes no practical difference. Knowing the truth matters
because it explains why a misplaced triple-quoted string silently does nothing.

In [ ]:
import dis

NEWLINE = chr(10)
QUOTES = chr(34) * 3

# Version A: real comments inside the function.
with_comments = (
    "def f():" + NEWLINE
    + "    # line one" + NEWLINE
    + "    # line two" + NEWLINE
    + "    return 1" + NEWLINE
)

# Version B: a triple-quoted string used as a block comment.
with_string = (
    "def f():" + NEWLINE
    + "    " + QUOTES + NEWLINE
    + "    line one" + NEWLINE
    + "    line two" + NEWLINE
    + "    " + QUOTES + NEWLINE
    + "    return 1" + NEWLINE
)

# Compile both and look at what each function object stored.
comments_function = compile(with_comments, "<demo>", "exec").co_consts[0]
string_function = compile(with_string, "<demo>", "exec").co_consts[0]

print("Constants in the COMMENT version:", comments_function.co_consts)
print("Constants in the STRING version: ", string_function.co_consts)

print("")
print("The triple-quoted string became the function's DOCSTRING, because")
print("it was the first statement. It was not discarded - it was stored.")
print("")
print("Placed anywhere else it would be built at runtime and thrown away,")
print("which costs a little and achieves nothing. Use # for comments.")

## Docstring conventions

Three styles dominate. Pick one and use it consistently — tooling depends on the
format being predictable.

In [ ]:
def google_style(price, quantity, tax_rate=0.18):
    """Calculate a total including tax.

    Args:
        price: Cost of a single item.
        quantity: Number of items.
        tax_rate: Tax rate as a decimal.

    Returns:
        The total with tax applied.

    Raises:
        ValueError: If quantity is negative.
    """
    if quantity < 0:
        raise ValueError("quantity cannot be negative")
    return price * quantity * (1 + tax_rate)


def numpy_style(price, quantity):
    """Calculate a total.

    Parameters
    ----------
    price : float
        Cost of a single item.
    quantity : int
        Number of items.

    Returns
    -------
    float
        The total cost.
    """
    return price * quantity


def sphinx_style(price, quantity):
    """Calculate a total.

    :param price: Cost of a single item.
    :param quantity: Number of items.
    :returns: The total cost.
    """
    return price * quantity


styles = [
    ("Google", "readable, most popular for new projects", google_style),
    ("NumPy", "common in scientific and data libraries", numpy_style),
    ("Sphinx", "the original, still widespread", sphinx_style),
]

print("Style    Used for                                   Lines")
print("-" * 62)
for name, used_for, function in styles:
    line_count = len(function.__doc__.strip().splitlines())
    print(name.ljust(8), used_for.ljust(42), line_count)

print("")
print("All three work. This course uses Google style for readability.")

## Good and bad comments, side by side

The examples below are all syntactically fine. The difference is whether they
help the next person.

In [ ]:
# Each entry is (verdict, the comment, why).
comment_examples = [
    ("BAD", "# increment i by 1", "restates the code exactly"),
    ("BAD", "# loop over the list", "the for line already says this"),
    ("BAD", "# fixed bug", "which bug? use version control instead"),
    ("GOOD", "# API rate limit is 10/sec, so throttle here", "external constraint"),
    ("GOOD", "# Using a list not a set: order matters downstream", "explains a choice"),
    ("GOOD", "# Workaround for upstream bug #4471, remove after v2.3", "temporary, with an exit condition"),
    ("GOOD", "# Binary search needs a sorted input - see sort above", "states a precondition"),
]

print("Verdict  Comment                                              Why")
print("-" * 100)
for verdict, comment, reason in comment_examples:
    print(verdict.ljust(8), comment.ljust(52), reason)

print("")
print("Test for any comment: does it say something the code CANNOT?")
print("If not, delete it - it will only rot as the code changes.")

## Special comments Python and its tools understand

Some comments are read by tooling even though Python's compiler discards them.

In [ ]:
special_comments = [
    ("# type: ignore", "mypy", "skip type checking on this line"),
    ("# noqa", "ruff / flake8", "skip all lint warnings on this line"),
    ("# noqa: E501", "ruff / flake8", "skip one specific warning"),
    ("# pragma: no cover", "coverage.py", "exclude from coverage reporting"),
    ("# fmt: off / on", "black / ruff", "stop and start auto-formatting"),
    ("# TODO: ...", "editors", "highlighted in most IDEs"),
    ("# FIXME: ...", "editors", "highlighted as more urgent than TODO"),
    ("# -*- coding: utf-8 -*-", "Python 2", "obsolete - UTF-8 is the default now"),
]

print("Comment                      Read by         Means")
print("-" * 78)
for comment, tool, meaning in special_comments:
    print(comment.ljust(29), tool.ljust(16), meaning)

print("")
print("Use these sparingly. Each one silences a check that exists for a reason.")

## Commenting style rules from PEP 8

These are mechanical and worth internalising now.

In [ ]:
# RULE 1: inline comments need two spaces before the #, then one space after.
value = 42  # correct spacing

# RULE 2: block comments sit at the same indentation as the code they describe.
for number in range(3):
    # This comment is indented with the code it explains.
    print("   value:", number)

# RULE 3: sentences start with a capital and read as English.
# Good: Calculate the running total before applying tax.
# Poor: calc tot b4 tax

# RULE 4: keep comments under the line-length limit, same as code.

print("")
print("PEP 8 comment rules:")
rules = [
    "Two spaces before an inline #, one space after",
    "Block comments align with the code they describe",
    "Write complete sentences, capitalised",
    "Update the comment whenever you change the code",
    "Delete commented-out code - version control remembers it",
]
for index, rule in enumerate(rules, start=1):
    print("   ", index, "-", rule)

## Takeaways

1. Comments are **discarded at compile time**; docstrings are **stored** in
   `__doc__` and readable at runtime.
2. A docstring must be the **first statement** of a module, function, class, or
   method — anywhere else it is a discarded string.
3. Comments explain **why**; docstrings explain **what** and **how to use it**.
4. Python has **no block comment syntax** — a triple-quoted string is an
   expression statement, not a comment.
5. A comment that restates the code is worse than no comment, because it rots.
6. Pick one docstring convention (Google, NumPy or Sphinx) and stay consistent.
7. Special comments like `# noqa` and `# type: ignore` are read by tools, not by
   Python itself.

## Try it yourself

1. Write a function with a docstring, then read it back with `__doc__` and with
   `help()`. What does `help()` add?
2. Move the docstring to the second line of the function. What does `__doc__`
   return now?
3. Find a comment in your own code that restates what the code does. Rewrite it
   to explain why, or delete it.
4. Run `print(dict.__doc__)` and `print(str.join.__doc__)`. How much of the
   standard library can you learn this way?